# 04 · Exploratory analysis
PCA / UMAP / clustering on the QC-filtered subset. Goal: see whether perturbation, batch or library size drives variation.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
import scanpy as sc
adata = sc.read_h5ad(P.processed / artifact("subset_adata", TAG))
if adata.n_obs > 20000:
    sc.pp.subsample(adata, n_obs=20000, random_state=0)
adata.obs["is_control"] = (adata.obs[PERT].astype(str) == CTRL).map({True: "control", False: "perturbed"})
sc.pp.pca(adata, n_comps=30)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20)
sc.tl.umap(adata, random_state=0)
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2, directed=False)

In [ ]:
top = adata.obs[PERT].astype(str).value_counts().drop(CTRL, errors="ignore").index[:8]
adata.obs["top_pert"] = adata.obs[PERT].astype(str).where(adata.obs[PERT].astype(str).isin(top), "other")
cols = ["is_control", "leiden", "top_pert"] + (["n_counts"] if "n_counts" in adata.obs else []) + ([BATCH] if BATCH in adata.obs else [])
sc.pl.umap(adata, color=cols, ncols=2)

In [ ]:
sc.pl.pca_variance_ratio(adata, n_pcs=30, log=True)
# Does the batch/library size separate cells more than perturbation?  Look at the UMAPs above and note it.
ct = pd.crosstab(adata.obs["leiden"], adata.obs["is_control"], normalize="index"); display(ct.round(2))

**Questions for your report:** (1) Do perturbed cells separate from controls globally, or only for strong perturbations? (2) Is there batch structure? (3) Which clusters are enriched for specific perturbations?